## Objective

Train a Linear Regression model using bootstrap to predict the best places for 
new oil wells.

Choose the region with the highest total profit for the selected oil wells.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from numpy.random import RandomState

There is a csv for each of three regions.

In [2]:
df0 = pd.read_csv('./datasets/geo_data_0.csv')
df1 = pd.read_csv('./datasets/geo_data_1.csv')
df2 = pd.read_csv('./datasets/geo_data_2.csv')

## Business parameters

In [3]:
# all values are in dolars

# total cost to build 200 wells
total_cost = 100000000

# cost per well
well_cost = total_cost / 200

# Profit per barrel is $4.5. Reserves volume unit is in thousands of barrels.
# profit per unit on 'product' column
unit_profit = 4500

## Data Cleaning

In [4]:
df0.head()

,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


In [5]:
df1.head()

,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305


In [6]:
df2.head()

,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746


f0, f1, f2 — locations significative characteristics.

product — reserves volume (thousands of barrels).

In [7]:
print(df0.info())
print()
print(df1.info())
print()
print(df2.info())


<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
None

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
None

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------  

There is no null values.

In [8]:
print(df0['id'].nunique())
print(df1['id'].nunique())
print(df2['id'].nunique())

99990
99996
99996


In [9]:
df0[df0['id'].duplicated(keep=False)].sort_values('id')

,id,f0,f1,f2,product
66136,74z30,1.084962,-0.312358,6.990771,127.643327
64022,74z30,0.741456,0.459229,5.153109,140.771492
51970,A5aEY,-0.180335,0.935548,-2.094773,33.020205
3389,A5aEY,-0.039949,0.156872,0.209861,89.249364
69163,AGS9W,-0.933795,0.116194,-3.655896,19.230453
42529,AGS9W,1.454747,-0.479651,0.683380,126.370504
931,HZww2,0.755284,0.368511,1.863211,30.681774
7530,HZww2,1.061194,-0.373969,10.430210,158.828695
63593,QcMuo,0.635635,-0.473422,0.862670,64.578675
1949,QcMuo,0.506563,-0.323775,-2.215583,75.496502


In [10]:
df1[df1['id'].duplicated(keep=False)].sort_values('id')

,id,f0,f1,f2,product
5849,5ltQ6,-3.435401,-12.296043,1.999796,57.085625
84461,5ltQ6,18.213839,2.191999,3.993869,107.813044
1305,LHZR0,11.170835,-1.945066,3.002872,80.859783
41906,LHZR0,-8.989672,-4.286607,2.009139,57.085625
2721,bfPNe,-9.494442,-5.463692,4.006042,110.992147
82178,bfPNe,-6.202799,-4.820045,2.995107,84.038886
47591,wt4Uk,-9.091098,-8.109279,-0.002314,3.179103
82873,wt4Uk,10.259972,-9.376355,4.994297,134.766305


In [11]:
df2[df2['id'].duplicated(keep=False)].sort_values('id')

,id,f0,f1,f2,product
45404,KUPhW,0.231846,-1.698941,4.990775,11.716299
55967,KUPhW,1.211150,3.176408,5.543540,132.831802
11449,VF7Jo,2.122656,-0.858275,5.746001,181.716817
49564,VF7Jo,-0.883115,0.560537,0.723601,136.233420
44378,Vcm5J,-1.229484,-2.439204,1.222909,137.968290
95090,Vcm5J,2.587702,1.986875,2.482245,92.327572
28039,xCHr8,1.633027,0.368135,-2.378367,6.120525
43233,xCHr8,-0.847066,2.101796,5.597130,184.388641


The duplicated values will be discarded, because there is no indication of 
which is the correct one. In a real situation, this would be verified with the 
team responsible for the data.

In [12]:
df0 = df0.drop_duplicates(subset='id', keep=False, ignore_index=True)
df1 = df1.drop_duplicates(subset='id', keep=False, ignore_index=True)
df2 = df2.drop_duplicates(subset='id', keep=False, ignore_index=True)

The column 'id' is irrelevant for model training and will be discarded.

In [13]:
df0 = df0.drop('id', axis=1)
df1 = df1.drop('id', axis=1)
df2 = df2.drop('id', axis=1)

## Functions

In [ ]:
def model_train(df):
    features = df.drop('product', axis=1)
    target = df['product']

    features_train, features_val, target_train, target_val = train_test_split(
        features,
        target,
        test_size=0.25,
        random_state=123,
    )

    scaler = StandardScaler()
    features_train = scaler.fit_transform(features_train)
    features_val = scaler.transform(features_val)

    model = LinearRegression()
    model.fit(features_train, target_train)
    predict_val = model.predict(features_val)

    predict_val_mean = predict_val.mean()
    rmse = mean_squared_error(target_val, predict_val) ** 0.5

    print(
        f'Mean predicted value: {predict_val_mean:.3f}\n'
        f'Root Mean Squared Error: {rmse:.3f}'
    )

    train_score = model.score(features_train, target_train)
    val_score = model.score(features_val, target_val)

    print(f'Train score: {train_score:.3f}')
    print(f'Validation score: {val_score:.3f}')

    return model, features_val, predict_val, target_val

def profit_estimate(predict_val, target_val):

    rand_gen = RandomState(123)

    val_df = pd.DataFrame({'predict': predict_val, 'target': target_val})
    val_df['predict_profit'] = (val_df['predict'] * unit_profit) - well_cost

    profits_list = []
    for i in range(1000):
        subsample = val_df.sample(500, replace=True, random_state = rand_gen)
        subsample = subsample.sort_values(by='predict', ascending=False)
        top_wells = subsample.head(200)
        sample_profit = top_wells['predict_profit'].sum()
        profits_list.append(sample_profit)

    profits = pd.Series(profits_list)
    mean_profit = profits.mean()
    lower_quantile = profits.quantile(0.025)
    upper_quantile = profits.quantile(0.975)

    loss_risk = (profits < 0).mean()

    print(f'Mean profit: {mean_profit:,.2f}')
    print(f'95% confidence interval: {lower_quantile:,.2f}; {upper_quantile:,.2f}')

    print(f'Loss risk: {loss_risk:.2%}')

## Model training

In [15]:
model_0, features_val_0, predict_val_0, target_val_0 = model_train(df0)

Mean predicted value: 92.734
Root Mean Squared Error: 37.556
Train score: 0.276
Validation score: 0.276


In [16]:
model_1, features_val_1, predict_val_1, target_val_1 = model_train(df1)

Mean predicted value: 68.610
Root Mean Squared Error: 0.894
Train score: 1.000
Validation score: 1.000


In [17]:
model_2, features_val_2, predict_val_2, target_val_2 = model_train(df2)

Mean predicted value: 95.002
Root Mean Squared Error: 39.941
Train score: 0.198
Validation score: 0.201


Model_1 has a much lower RMSE and much higher R².

## Profit estimate

### Profit estimate with all models.

In [18]:
profit_estimate(predict_val_0, target_val_0)

Mean profit: 3,879,930.73
95% interval: 1,642,218.02; 6,121,504.56
Loss risk: 0.10%


In [19]:
profit_estimate(predict_val_1, target_val_1)

Mean profit: 4,269,681.91
95% interval: 336,670.19; 8,450,766.84
Loss risk: 1.40%


In [20]:
profit_estimate(predict_val_2, target_val_2)

Mean profit: 2,923,498.58
95% interval: 911,367.78; 4,987,169.52
Loss risk: 0.00%
